In [44]:
import pandas as pd 

In [45]:
df = pd.read_csv("pathpilot_v1_filtered.csv")
df.head()

,course_id,course_title,url,platform,subject,level,num_subscribers,num_reviews,rating,num_lectures,...,language,tags,course_type,learning_path,playlist_id,skill_level_score,popularity_score,last_updated,carer_relevance_score,career_relevance_score
0,CRS_0001,Python for Everybody Specialization,https://www.coursera.org/specializations/python,Coursera,Python Programming,Beginner,0.184553,0.643222,0.8,96,...,English,"python,beginner,data structures,web scraping,d...",Video,Python Mastery Path,PL_011,1,0.762149,2024,0.537829,0.537829
1,CRS_0002,The Complete Python Bootcamp From Zero to Hero,https://www.udemy.com/course/complete-python-b...,Udemy,Python Programming,Beginner,0.276867,0.964854,0.4,155,...,English,"python,oop,decorators,generators,scripting",Video,Python Mastery Path,PL_011,1,0.895047,2024,0.597126,0.597126
2,CRS_0003,Machine Learning Specialization,https://www.coursera.org/specializations/machi...,Coursera,Machine Learning,Beginner,0.138395,0.361794,1.0,95,...,English,"supervised learning,regression,classification,...",Video,ML Engineer Path,PL_009,1,0.620866,2024,0.463060,0.463060
3,CRS_0004,Deep Learning Specialization,https://www.coursera.org/specializations/deep-...,Coursera,Deep Learning,Intermediate,0.130702,0.351743,1.0,120,...,English,"neural networks,backprop,cnn,rnn,hyperparamete...",Video,ML Engineer Path,PL_009,2,0.609843,2024,0.455650,0.455650
4,CRS_0005,Natural Language Processing Specialization,https://www.coursera.org/specializations/natur...,Coursera,Natural Language Processing,Intermediate,0.061466,0.130621,0.8,120,...,English,"nlp,transformers,attention,sentiment,named ent...",Video,NLP & LLM Engineer Path,PL_010,2,0.367334,2024,0.287635,0.287635


In [49]:
df.drop(columns=['carer_relevance_score'], inplace=True)

In [53]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4859 entries, 0 to 4858
Data columns (total 21 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   course_id               4859 non-null   str    
 1   course_title            4859 non-null   str    
 2   url                     4859 non-null   str    
 3   platform                4859 non-null   str    
 4   subject                 4859 non-null   str    
 5   level                   4859 non-null   str    
 6   num_subscribers         4859 non-null   float64
 7   num_reviews             4859 non-null   float64
 8   rating                  4859 non-null   float64
 9   num_lectures            4859 non-null   int64  
 10  content_duration        4859 non-null   float64
 11  instructor              4859 non-null   str    
 12  language                4859 non-null   str    
 13  tags                    4859 non-null   str    
 14  course_type             4859 non-null   str    
 15

In [52]:
df.to_csv("D:\Study\Project\PathPilot-AI\data\processed\pathpilot_v1_filtered.csv", index=True)

In [54]:
text_cols = ["course_title", "subject", "level", "tags", "learning_path", "instructor"]

for col in text_cols:
    df[col] = df[col].fillna("")

In [55]:
df["combined_features"] = (
    df["course_title"] + " " +
    df["subject"] + " " +
    df["level"] + " " +
    df["tags"] + " " +
    df["learning_path"] + " " +
    df["instructor"]
)

In [56]:
from sklearn.feature_extraction.text import TfidfVectorizer

tdifd = TfidfVectorizer(stop_words="english")
tdifd_matrix = tdifd.fit_transform(df['combined_features'])

In [57]:
tdifd_matrix.shape

(4859, 2356)

In [58]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_sim = cosine_similarity(tdifd_matrix, tdifd_matrix)

In [59]:
cosine_sim.shape

(4859, 4859)

In [60]:
indices = pd.Series(df.index, index=df['course_title']).drop_duplicates()

In [61]:
indices

course_title
Python for Everybody Specialization                                                            0
The Complete Python Bootcamp From Zero to Hero                                                 1
Machine Learning Specialization                                                                2
Deep Learning Specialization                                                                   3
Natural Language Processing Specialization                                                     4
                                                                                            ... 
Business & Product Thinking Topic 14 in Business & Product Thinking – Deep Dive Guide       4854
Business & Product Thinking Business & Product Thinking Topic 14 – Ultimate Course          4855
In-Depth Business & Product Thinking Business & Product Thinking Topic 4                    4856
Business & Product Thinking Topic 2 in Business & Product Thinking – Comprehensive Guide    4857
Mastering Busines

In [78]:
df["title_clean"] = df["course_title"].str.lower().str.strip()

In [79]:
import difflib

def find_best_match(user_input, df):
    user_input = user_input.lower().strip()

    # 1. Exact clean match
    exact_matches = df[df["title_clean"] == user_input]
    if not exact_matches.empty:
        return exact_matches.index[0]

    # 2. Partial contains match
    partial_matches = df[df["title_clean"].str.contains(user_input, na=False)]
    if not partial_matches.empty:
        return partial_matches.index[0]

    # 3. Fuzzy match
    possible_titles = df["title_clean"].tolist()
    close_matches = difflib.get_close_matches(user_input, possible_titles, n=1, cutoff=0.4)

    if close_matches:
        matched_title = close_matches[0]
        return df[df["title_clean"] == matched_title].index[0]

    return None

In [85]:
def recommend_courses(course_title, top_n=5):
    idx = find_best_match(course_title, df)

    if idx is None:
        return "Course not found!"

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:top_n+1]

    course_indices = [i[0] for i in sim_scores]

    return df[[
        "course_title",
        "subject",
        "level",
        "learning_path",
        "career_relevance_score"
    ]].iloc[course_indices]

In [86]:
recommend_courses("Machine", top_n=5)

,course_title,subject,level,learning_path,career_relevance_score
3935,Ultimate Ensemble Learning for Machine Learning,Machine Learning,Intermediate,ML Engineer Path,0.159008
3902,Comprehensive Machine Learning Logistic Regres...,Machine Learning,All Levels,ML Engineer Path,0.157782
3,Deep Learning Specialization,Deep Learning,Intermediate,ML Engineer Path,0.455650
3924,Mastering Logistic Regression (Machine Learning),Machine Learning,Beginner,ML Engineer Path,0.210948
3919,Masterclass Meta Learning for Machine Learning,Machine Learning,Beginner,ML Engineer Path,0.165892


In [87]:
level_map = {
    "Beginner": 1,
    "Intermediate": 2,
    "Advanced": 3
}

df["level_num"] = df["level"].map(level_map)

In [88]:
def recommend_smart(course_title, top_n=5):
    idx = find_best_match(course_title, df)

    if idx is None:
        return "Course not found!"

    input_level = df.loc[idx, "level_num"]

    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:20]

    candidate_data = []

    for i, sim_score in sim_scores:
        row = df.iloc[i]

        if row["level_num"] >= input_level and row["level_num"] <= input_level + 1:
            final_score = (
                0.5 * sim_score +
                0.3 * row["career_relevance_score"] +
                0.2 * row["skill_level_score"]
            )

            candidate_data.append({
                "course_title": row["course_title"],
                "subject": row["subject"],
                "level": row["level"],
                "learning_path": row["learning_path"],
                "career_relevance_score": row["career_relevance_score"],
                "skill_level_score": row["skill_level_score"],
                "similarity_score": sim_score,
                "final_score": final_score
            })

    result_df = pd.DataFrame(candidate_data)
    return result_df.sort_values(by="final_score", ascending=False).head(top_n)

In [89]:
recommend_smart("Python for", top_n=5)

,course_title,subject,level,learning_path,career_relevance_score,skill_level_score,similarity_score,final_score
2,Python for Data Structures,Python Programming,Intermediate,Python Mastery Path,0.201899,2,0.499621,0.710380
10,Python for Generators,Python Programming,Intermediate,Python Mastery Path,0.279980,2,0.438637,0.703312
14,Python Networking Bootcamp,Python Programming,Intermediate,Python Mastery Path,0.286590,2,0.434508,0.703231
7,Python Regex in Practice,Python Programming,Intermediate,Python Mastery Path,0.176133,2,0.463407,0.684543
5,Mastering Web APIs (Python Programming),Python Programming,Intermediate,Python Mastery Path,0.053521,2,0.470806,0.651459


In [93]:
df.to_csv("C:/Users/Preneel/Jupyter Notebook/Projects/Path Pilot AI/pathpilot_v1_filtered.csv", index=False)

In [94]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 4859 entries, 0 to 4858
Data columns (total 24 columns):
 #   Column                  Non-Null Count  Dtype  
---  ------                  --------------  -----  
 0   course_id               4859 non-null   str    
 1   course_title            4859 non-null   str    
 2   url                     4859 non-null   str    
 3   platform                4859 non-null   str    
 4   subject                 4859 non-null   str    
 5   level                   4859 non-null   str    
 6   num_subscribers         4859 non-null   float64
 7   num_reviews             4859 non-null   float64
 8   rating                  4859 non-null   float64
 9   num_lectures            4859 non-null   int64  
 10  content_duration        4859 non-null   float64
 11  instructor              4859 non-null   str    
 12  language                4859 non-null   str    
 13  tags                    4859 non-null   str    
 14  course_type             4859 non-null   str    
 15